# Promotion forecast lab

**All data are synthetic.** This is an authorized adaptation of the covariate forecasting method in *Getting Started with Chronos-2*, by Amazon Science / the Chronos authors. It does not reproduce Rossmann or any electricity dataset, and does not download or redistribute upstream data.

Source: [official notebook](https://github.com/amazon-science/chronos-forecasting/blob/10afa9ebe016e514f9d7dc1aa873f66af57e116b/notebooks/chronos-2-quickstart.ipynb), commit `10afa9ebe016e514f9d7dc1aa873f66af57e116b`. The unmodified original and provenance are retained in `source/`; the supplied provenance lists the notebook license as “not supplied”.

Model: [AutoGluon Chronos-2 Small](https://huggingface.co/autogluon/chronos-2-small), **Apache-2.0**, pinned revision `ddec01313e50b6bc58ebaa92ede81bc24a3d9f9a`. Inference uses CPU float32 and four threads. Set `CHRONOS_MODEL_PATH` to the predownloaded snapshot, or have this exact revision in the local Hugging Face cache. No packages or models are downloaded by this workflow.

The upstream retail example supplies historical targets and known future covariates and compares covariate-enabled and target-only forecasts. We retain that analysis using promotion and sine-encoded weekday features with synthetic daily sales. This is a scenario forecast, **not a causal estimate or a production-accuracy claim**.

## 1. Prepare synthetic history and known future covariates

The fixture contains 256 history days and a selectable future promotion interval, clipped to the horizon. Day 0 means the first forecast day. The verifier supplies `PROJECT_ROOT`; no current-directory assumptions or machine-specific paths are needed.

In [ ]:
from pathlib import Path
import os
import sys

project_root = Path(globals().get("PROJECT_ROOT", os.environ.get("PROJECT_ROOT", "."))).resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from forecast_core import make_fixture, predict, seasonal_baseline, assemble_results

parameters = dict(seed=42, horizon=28, promotion_start=7, promotion_days=7)
data = make_fixture(**parameters)
print("SYNTHETIC history:", len(data["context"]), "days")
print("Known future promotion offsets:", np.flatnonzero(data["future_promotion"]).tolist())
print(pd.DataFrame({
    "forecast_day": np.arange(parameters["horizon"]),
    "known_promotion": data["future_promotion"],
    "known_weekday_sine": data["future_weekday"],
}).head(10).to_string(index=False))

## 2. Run real Chronos inference and the seasonal baseline

The prediction API accepts historical sales as the **only target**. Future sales are excluded even from this input dictionary. Quantile levels are 0.1, 0.5 and 0.9. The main forecast uses known covariates; a second real inference call reproduces the upstream target-only comparison. The seasonal-7 baseline simply repeats the last seven historical observations.

In [ ]:
inference_data = {key: values for key, values in data.items() if key != "actual"}
predictions = predict(inference_data, use_covariates=True)
without_covariates = predict(inference_data, use_covariates=False)
baseline = seasonal_baseline(data["context"], parameters["horizon"])
print("Completed actual Chronos-2 Small inference with and without covariates.")
print("Forecast shape:", predictions["forecast"].shape)

## 3. Evaluate and persist results

MAE is in synthetic sales units. Observed p10–p90 coverage is the fraction of the held-out synthetic observations inside the model interval. Its nominal level is 80%, but **the interval is not calibrated on this fixture**. A single synthetic seed/horizon cannot establish production accuracy, and covariates need not always improve forecasts. `results.json` is freshly written in the execution directory using finite JSON values.

In [ ]:
import json

results = assemble_results(data, predictions, baseline, parameters)
results["without_covariates"] = {
    "predictions": {key: values.tolist() for key, values in without_covariates.items()},
    "mae": float(np.mean(np.abs(data["actual"] - without_covariates["forecast"]))),
}
results["interpretation"] = (
    "Synthetic scenario forecast, not a causal estimate or a production-accuracy claim. "
    "Nominal 80% p10-p90 intervals are not calibrated on this fixture."
)
output = Path.cwd() / "results.json"
output.write_text(json.dumps({"results": results}, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(pd.DataFrame([
    {"method": "Chronos with covariates", "mae": results["metrics"]["mae"]},
    {"method": "Chronos without covariates", "mae": results["without_covariates"]["mae"]},
    {"method": "Seasonal-7", "mae": results["metrics"]["baseline_mae"]},
]).to_string(index=False))
print("Observed p10-p90 coverage:", results["metrics"]["coverage"])
print("Fresh artifact:", output.name)